# P2 DL fundamentals: Kaggle runner

Thin wrapper: all logic lives in `src/dlfund`.

Settings: **Accelerator GPU T4 x1** (or P100), **Internet ON**. Use *Save Version → Save & Run All (commit)* so it keeps running after you close the tab.

Every run is resumable within a session: if a cell stops, rerun it and finished (seed, model) pairs are skipped.

At the end, download `p2_results.zip` from the Output tab (metrics, per-class tables and figures; no weights).

In [ ]:
!git clone --depth 1 https://github.com/JDevender007/Artificial_Intelligence.git
%cd Artificial_Intelligence/02-vision-from-scratch
!pip install -q -e . --no-deps && pip install -q tabulate

In [ ]:
import torch

print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
!python scripts/download_data.py --data-dir /kaggle/working/data

## 1. MNIST: linear / mlp / cnn × 3 seeds

In [ ]:
!python scripts/run_experiment.py --config configs/mnist.yaml --set output_dir=/kaggle/working/outputs/mnist data.data_dir=/kaggle/working/data
print(open('/kaggle/working/outputs/mnist/results.md').read())

## 2. CIFAR-10: linear / mlp / cnn / resnet18 × 3 seeds
The longest step. ResNet-18 downloads its ImageNet weights automatically.

In [ ]:
!python scripts/run_experiment.py --config configs/cifar10.yaml --set output_dir=/kaggle/working/outputs/cifar10 data.data_dir=/kaggle/working/data
print(open('/kaggle/working/outputs/cifar10/results.md').read())

## 3. CNN ablations on CIFAR-10 (see `configs/ablations.md`)

In [ ]:
!python scripts/run_experiment.py --config configs/cifar10.yaml --set output_dir=/kaggle/working/outputs/abl_no_aug data.data_dir=/kaggle/working/data name=no_aug "models=[cnn]" data.augment=false
!python scripts/run_experiment.py --config configs/cifar10.yaml --set output_dir=/kaggle/working/outputs/abl_no_sched data.data_dir=/kaggle/working/data name=no_sched "models=[cnn]" train.scheduler=none train.lr=0.01
!python scripts/run_experiment.py --config configs/cifar10.yaml --set output_dir=/kaggle/working/outputs/abl_no_amp data.data_dir=/kaggle/working/data name=no_amp "models=[cnn]" train.amp=false

In [ ]:
!python scripts/compare_ablations.py --main /kaggle/working/outputs/cifar10 --glob '/kaggle/working/outputs/abl_*' --out /kaggle/working/outputs/ablations.md
print(open('/kaggle/working/outputs/ablations.md').read())

## 4. Package results (no checkpoints)

In [ ]:
!cd /kaggle/working && zip -qr p2_results.zip outputs -x '*/checkpoints/*' '*/tb/*' && ls -lh p2_results.zip